# MacroTrader Layer

This NB is for MacroTrader Layer and it's base models

## Installations

In [1]:
# !pip uninstall gymnasium shimmy stable-baselines3 -y
!pip install gym==0.26.0 shimmy>=0.2.1 stable-baselines3 torch
!pip install h5py

In [2]:
import pandas as pd

# Set option to display all columns
pd.set_option('display.max_columns', None)

In [ ]:
# Install TA-Lib using conda
!conda install -c conda-forge ta-lib -y

## Data

#### One script to pull data + merge data

In [4]:
import sys
sys.path.append('Data_Script')
from Data_Script.fetch_merge_data import fetch_and_merge_data

data = fetch_and_merge_data('AAPL','2024-07-07','2024-08-07')

data.head()

/home/ec2-user/SageMaker/Data_Script/fetch_merge_data.py:107: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  quotes = df.resample('min').agg({


Total Quote Process Time:  164.4353063106537  seconds.
Merged data saved to merged_data_AAPL_2024-07-07_2024-08-07.csv


,timestamp,datetime,open,high,low,close,volume,ask_price,ask_size,bid_price,bid_size
0,1720425600000,2024-07-08 08:00:00,227.60,227.6,226.61,226.82,3947.0,0.0,0,0.0,0
1,1720425660000,2024-07-08 08:01:00,226.90,226.9,226.90,226.90,1914.0,0.0,0,0.0,0
2,1720425720000,2024-07-08 08:02:00,226.87,227.0,226.87,227.00,1325.0,0.0,0,0.0,0
3,1720425780000,2024-07-08 08:03:00,227.00,227.1,227.00,227.10,1037.0,0.0,0,0.0,0
4,1720425840000,2024-07-08 08:04:00,227.20,227.2,227.20,227.20,1819.0,0.0,0,0.0,0


Preparing new data for training...

#### Adding Tech Indicators

In [20]:
data_training = data.copy()

In [21]:
# Define Tech Indicators
import pandas as pd
import numpy as np
import talib as ta
import numpy as np
import math

class TechnicalIndicators:
    def __init__(self, data):
        """
        Initializes the TechnicalIndicators class with the provided data.

        Parameters:
        - data: A pandas DataFrame containing financial data with columns like 'close', 'high', 'low', 'volume', etc.
        """
        self.data = data

    def add_momentum_indicators(self):
        """
        Adds momentum indicators such as RSI, MACD, and Stochastic Oscillator to the data.
        """
        # Relative Strength Index (RSI)
        self.data['RSI'] = ta.RSI(self.data['close'], timeperiod=14)
        
        # Moving Average Convergence Divergence (MACD)
        self.data['MACD'], self.data['MACD_signal'], self.data['MACD_hist'] = ta.MACD(
            self.data['close'], fastperiod=12, slowperiod=26, signalperiod=9)
        
        # Stochastic Oscillator
        self.data['Stoch_k'], self.data['Stoch_d'] = ta.STOCH(
            self.data['high'], self.data['low'], self.data['close'], fastk_period=14, slowk_period=3, slowd_period=3)

    def add_volume_indicators(self):
        """
        Adds volume indicators such as On-Balance Volume (OBV) to the data.
        """
        # On-Balance Volume (OBV)
        self.data['OBV'] = ta.OBV(self.data['close'], self.data['volume'])
        
    def add_TC(self):
        window_size = 5
        self.data['mid_price'] = (self.data['high'] + self.data['low']) / 2
        self.data["mean_vol"] = self.data['mid_price'].pct_change().rolling(window=window_size).mean()
        self.data["mean_liq"] = self.data['volume'].rolling(window=window_size).mean()
        self.data = self.data.iloc[35:,:]


        # AC Calculation -> 
        x0 = 5000  # Initial number of shares to trade
        T = 1.0    # Total time horizon (e.g., 1 day)
        N = min(x0, 2400)  # Number of discrete time intervals
        eta = 0.0000001    # Temporary/permanent impact coefficient
        sigma = 0.02       # Volatility of the asset
        lambda_ = 0.1      # Risk aversion parameter

        # Time interval
        dt = 1

        # Cost function for the Almgren-Chriss model
        def cost_function(x, eta, sigma, x0=10):
            x_cumsum = np.cumsum(x)
            x_half = x / 2

            temp_cost = np.sum(eta * (x**2) / dt)
            perm_cost = np.sum(eta * x * (x0 - x_cumsum + x_half))
            var_cost = np.sum(lambda_ * sigma**2 * (x**2) * dt)

            return (1 / (temp_cost + perm_cost + var_cost) / x0) * 10**(math.log10(x0) * 4 - 6)

        def almgren(row):
            x0 = 5000
            N = min(x0, 2400)

            eta = 1 / row['mean_liq'] if row['mean_liq'] != 0 else 0.0000001
            sigma = row['mean_vol']

            # Initial guess: trade evenly across all intervals
            x_init = np.ones(N) * (x0 / N)

            return cost_function(x_init, eta, sigma, x0) / x0
        
        self.data['transaction_cost'] = self.data.apply(almgren, axis=1)

    def add_volatility_indicators(self):
        """
        Adds volatility indicators such as Bollinger Bands and Average True Range (ATR) to the data.
        """
        # Bollinger Bands
        self.data['Upper_BB'], self.data['Middle_BB'], self.data['Lower_BB'] = ta.BBANDS(self.data['close'], timeperiod=20)
        
        # Average True Range (ATR) for different periods
        self.data['ATR_1'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=1)
        self.data['ATR_2'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=2)
        self.data['ATR_5'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=5)
        self.data['ATR_10'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=10)
        self.data['ATR_20'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=20)
        
    def add_volatility(self, window=15):
        """
        Adds dynamic volatility calculation using log returns and a rolling window.

        Parameters:
        - window: The rolling window size for calculating volatility (default is 15).
        """
        # Log returns
        self.data['log_return'] = np.log(self.data['close'] / self.data['close'].shift(1))
        
        # Rolling volatility
        self.data['volatility'] = self.data['log_return'].rolling(window=window).std() * np.sqrt(window)

    def add_trend_indicators(self):
        """
        Adds trend indicators such as ADX, +DI, -DI, and CCI to the data.
        """
        # Average Directional Index (ADX)
        self.data['ADX'] = ta.ADX(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        
        # Plus Directional Indicator (+DI)
        self.data['+DI'] = ta.PLUS_DI(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        
        # Minus Directional Indicator (-DI)
        self.data['-DI'] = ta.MINUS_DI(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        
        # Commodity Channel Index (CCI)
        self.data['CCI'] = ta.CCI(self.data['high'], self.data['low'], self.data['close'], timeperiod=5)
        
    def add_5_min_indicators(self):
        # code to add 5 mins volume, volatility, TC
        self.data['5_min_volatility'] = self.data['volatility'].transform(lambda x: x.rolling(window=5).std())
        self.data['5_min_volume'] = self.data['volume'].transform(lambda x: x.rolling(window=5).sum())
        self.data['5_min_TC'] = self.data['transaction_cost'].shift(5)

    def add_other_indicators(self):
        """
        Adds other indicators such as DLR, TWAP, VWAP, market liquidity, and expected price to the data.
        """
        # Daily Log Returns (DLR)
        self.data['DLR'] = np.log(self.data['close'] / self.data['close'].shift(1))
        
        # Time-Weighted Average Price (TWAP)
        self.data['TWAP'] = self.data['close'].expanding().mean()
        
        # Volume-Weighted Average Price (VWAP)
        self.data['VWAP'] = (self.data['volume'] * (self.data['high'] + self.data['low']) / 2).cumsum() / self.data['volume'].cumsum()
        
        # Market Liquidity
        self.data['market_liquidity'] = self.data['bid_size'] + self.data['ask_size']
        
        # Expected Price
        self.data['expected_price'] = self.data['bid_price']
        

    def add_all_indicators(self):
        """
        Adds all the defined indicators to the data.
        
        Returns:
        - The updated DataFrame with all indicators added.
        """
        self.add_momentum_indicators()
        self.add_volume_indicators()
        self.add_volatility_indicators()
        self.add_trend_indicators()
        self.add_other_indicators()
        self.add_volatility()
        self.add_TC()
        self.add_5_min_indicators()
        return self.data

In [22]:
# Create an instance of TechnicalIndicators
indicators = TechnicalIndicators(data_training)

# Add all indicators
data_with_indicators = indicators.add_all_indicators()

# Discard NaN values 
data_training = data_with_indicators.iloc[35:]

# Display the updated data
data_training.head(10)

/tmp/ipykernel_27772/1891636869.py:82: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['transaction_cost'] = self.data.apply(almgren, axis=1)
/tmp/ipykernel_27772/1891636869.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['5_min_volatility'] = self.data['volatility'].transform(lambda x: x.rolling(window=5).std())
/tmp/ipykernel_27772/1891636869.py:130: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

,timestamp,datetime,open,high,low,close,volume,ask_price,ask_size,bid_price,bid_size,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ATR_2,ATR_5,ATR_10,ATR_20,ADX,+DI,-DI,CCI,DLR,TWAP,VWAP,market_liquidity,expected_price,log_return,volatility,mid_price,mean_vol,mean_liq,transaction_cost,5_min_volatility,5_min_volume,5_min_TC
70,1720432740000,2024-07-08 09:59:00,227.16,227.16,227.16,227.16,384.0,227.21,17,227.13,12,42.582713,-0.053827,-0.044779,-0.009049,10.562766,9.623502,6587.0,227.503063,227.2585,227.013937,0.02,0.029639,0.035085,0.044326,0.056637,15.080945,39.321896,51.500034,-10.752688,0.000088,227.253521,227.269340,29,227.13,0.000088,0.001054,227.16,-0.000009,614.4,0.001228,0.000079,3072.0,0.001077
71,1720433040000,2024-07-08 10:04:00,227.23,227.23,227.23,227.23,233.0,227.23,57,227.17,10,48.455459,-0.047199,-0.045263,-0.001936,24.041585,14.656334,6820.0,227.487690,227.2505,227.013310,0.07,0.049820,0.042068,0.046894,0.057306,14.105253,45.204103,46.507560,136.524823,0.000308,227.253194,227.269230,67,227.17,0.000308,0.000957,227.23,0.000070,573.0,0.001145,0.000124,2865.0,0.000912
72,1720433520000,2024-07-08 10:12:00,227.23,227.25,227.23,227.25,1904.0,227.25,14,227.20,5,50.028140,-0.039872,-0.044185,0.004313,51.378446,28.660933,8724.0,227.478264,227.2455,227.012736,0.02,0.034910,0.037654,0.044204,0.055440,13.224409,46.791217,45.160510,95.771144,0.000088,227.253151,227.268575,19,227.20,0.000088,0.000868,227.24,0.000062,846.6,0.001692,0.000149,4233.0,0.000922
73,1720433940000,2024-07-08 10:19:00,227.25,227.25,227.25,227.25,250.0,227.25,3,227.20,13,50.028140,-0.033677,-0.042083,0.008406,79.448622,51.622884,8724.0,227.457761,227.2375,227.017239,0.00,0.017455,0.030124,0.039784,0.052668,12.406482,46.791217,45.160510,68.862275,0.000000,227.253108,227.268520,16,227.20,0.000000,0.000862,227.25,0.000053,765.6,0.001530,0.000142,3828.0,0.001059
74,1720434240000,2024-07-08 10:24:00,227.25,227.25,227.25,227.25,315.0,227.25,10,227.21,36,50.028140,-0.028440,-0.039355,0.010914,95.238095,75.355054,8724.0,227.427960,227.2280,227.028040,0.00,0.008727,0.024099,0.035806,0.050035,11.646979,46.791217,45.160510,58.333333,0.000000,227.253067,227.268452,46,227.21,0.000000,0.000669,227.25,0.000101,617.2,0.001233,0.000143,3086.0,0.001326
75,1720434300000,2024-07-08 10:25:00,227.25,227.25,227.25,227.25,269.0,227.25,8,227.24,7,50.028140,-0.024013,-0.036286,0.012273,100.000000,91.562239,8724.0,227.348809,227.2130,227.077191,0.00,0.004364,0.019279,0.032225,0.047533,10.941726,46.791217,45.160510,55.555556,0.000000,227.253026,227.268394,15,227.24,0.000000,0.000620,227.25,0.000079,594.2,0.001187,0.000144,2971.0,0.001228
76,1720434360000,2024-07-08 10:26:00,227.25,227.25,227.25,227.25,318.0,227.25,0,227.24,0,50.028140,-0.020270,-0.033083,0.012813,100.000000,98.412698,8724.0,227.312733,227.2055,227.098267,0.00,0.002182,0.015423,0.029003,0.045156,10.286849,46.791217,45.160510,41.666667,0.000000,227.252987,227.268326,0,227.24,0.000000,0.000620,227.25,0.000018,611.2,0.001221,0.000127,3056.0,0.001145
77,1720434420000,2024-07-08 10:27:00,227.25,227.25,227.25,227.25,255.0,227.25,3,227.24,2,50.028140,-0.017107,-0.029888,0.012780,100.000000,100.000000,8724.0,227.303419,227.2030,227.102581,0.00,0.001091,0.012339,0.026102,0.042899,9.678748,46.791217,45.160510,0.000000,0.000000,227.252949,227.268272,5,227.24,0.000000,0.000620,227.25,0.000009,281.4,0.000562,0.000105,1407.0,0.001692
78,1720434540000,2024-07-08 10:29:00,227.26,227.26,227.26,227.26,491.0,227.33,23,227.24,11,51.189716,-0.013637,-0.026638,0.013001,100.000000,100.000000,9215.0,227.289219,227.2000,227.110781,0.01,0.005545,0.011871,0.024492,0.041254,9.282327,47.966709,44.162821,166.666667,0.000044,227.253038,227.268226,34,227.24,0.000044,0.000615,227.26,0.000009,329.6,0.000659,0.000022,1648.0,0.001530
79,1720434780000,2024-07-08 10:33:00,227.24,227.24,227.24,227.24,549.0,227.24,143,227.21,18,48.749080,-0.012357,-0.023782,0.011424,94.871795,98.290598,8666.0,227.292907,227.2020,227

In [23]:
len(data_training)

17647

#### Adding forecasts

In [ ]:
from joblib import Parallel, delayed
from statsmodels.tsa.api import ARIMA, ExponentialSmoothing
from tqdm import tqdm  # Import tqdm for progress tracking
import time

def forecast_row(idx, data, forecast_steps, columns, window_size):
    row_forecasts = {}
    row_forecasts['timestamp'] = data.index[idx]

    for indicator, column in columns.items():
        for key, (steps, freq) in forecast_steps.items():
            if indicator not in key:
                continue
            try:
                # Use a sliding window
                start_idx = max(0, idx - window_size)
                series = data[column].iloc[start_idx:idx+1]
                if len(series) < 2:
                    row_forecasts[f'forecast_{indicator}_{key}'] = None
                    continue

                if indicator in ['open', 'high', 'low', 'close', 'transaction_cost']:
                    model = ExponentialSmoothing(series, trend='add', seasonal=None)
                elif indicator == 'volatility':
                    model = ARIMA(series, order=(5, 1, 0))
                elif indicator == 'volume':
                    shift = 1 if series.min() <= 0 else 0
                    transformed_series = np.log(series + shift + 1)
                    model = ExponentialSmoothing(transformed_series, trend='add', seasonal=None)
                model_fit = model.fit()

                forecast_values = model_fit.forecast(steps=steps)
                row_forecasts[f'forecast_6Hr_{indicator}'] = forecast_values.iloc[-1]

            except Exception as e:
                row_forecasts[f'forecast_6Hr_{indicator}'] = None

    return row_forecasts

def train_and_forecast_parallel(data, forecast_steps, window_size, n_jobs=-1):
    columns = {
        'open': 'open',
        'high': 'high',
        'low': 'low',
        'close': 'close',
        'volatility': 'volatility',
        'volume': 'volume',
        'transaction_cost': 'transaction_cost'
    }
    
    print('Starting...')
    # Add tqdm progress bar
    results = Parallel(n_jobs=n_jobs)(delayed(forecast_row)(idx, data, forecast_steps, columns, window_size) for idx in tqdm(range(len(data)), desc="Processing rows"))

    forecast_results = pd.DataFrame(results).set_index('timestamp')
    return forecast_results

# forecast steps
forecast_steps = {
    'open': (360, '1T'),
    'high': (360, '1T'),
    'low': (360, '1T'),
    'close': (360, '1T'),
    'volatility': (360, '1T'),
    'volume': (360, '1T'),
    'transaction_cost': (360, '1T')
}

# sliding window size (last 100 observations)
window_size = 100

start_time = time.time()

forecasted_data = train_and_forecast_parallel(data_training, forecast_steps, window_size)

end_time = time.time()
elapsed_time = end_time - start_time

# # Fill/drop NaN values
# forecasted_data = forecasted_data.fillna(method='ffill').fillna(method='bfill')

# Combine the forecasted data
combined_data = data_training.join(forecasted_data)
print(combined_data.head())
print(f"Elapsed time: {elapsed_time:.2f} seconds")


In [25]:
combined_data.drop(['forecast_open_open',
       'forecast_high_high', 'forecast_low_low', 'forecast_close_close',
       'forecast_volatility_volatility', 'forecast_volume_volume',
       'forecast_transaction_cost_transaction_cost'], axis=1, inplace=True)

In [26]:
combined_data.head()

,timestamp,datetime,open,high,low,close,volume,ask_price,ask_size,bid_price,bid_size,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ATR_2,ATR_5,ATR_10,ATR_20,ADX,+DI,-DI,CCI,DLR,TWAP,VWAP,market_liquidity,expected_price,log_return,volatility,mid_price,mean_vol,mean_liq,transaction_cost,5_min_volatility,5_min_volume,5_min_TC,forecast_6Hr_open,forecast_6Hr_high,forecast_6Hr_low,forecast_6Hr_close,forecast_6Hr_volatility,forecast_6Hr_volume,forecast_6Hr_transaction_cost
70,1720432740000,2024-07-08 09:59:00,227.16,227.16,227.16,227.16,384.0,227.21,17,227.13,12,42.582713,-0.053827,-0.044779,-0.009049,10.562766,9.623502,6587.0,227.503063,227.2585,227.013937,0.02,0.029639,0.035085,0.044326,0.056637,15.080945,39.321896,51.500034,-10.752688,0.000088,227.253521,227.269340,29,227.13,0.000088,0.001054,227.16,-0.000009,614.4,0.001228,0.000079,3072.0,0.001077,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71,1720433040000,2024-07-08 10:04:00,227.23,227.23,227.23,227.23,233.0,227.23,57,227.17,10,48.455459,-0.047199,-0.045263,-0.001936,24.041585,14.656334,6820.0,227.487690,227.2505,227.013310,0.07,0.049820,0.042068,0.046894,0.057306,14.105253,45.204103,46.507560,136.524823,0.000308,227.253194,227.269230,67,227.17,0.000308,0.000957,227.23,0.000070,573.0,0.001145,0.000124,2865.0,0.000912,252.424985,252.424985,252.424985,252.424985,NaN,-173.795700,-0.019360
72,1720433520000,2024-07-08 10:12:00,227.23,227.25,227.23,227.25,1904.0,227.25,14,227.20,5,50.028140,-0.039872,-0.044185,0.004313,51.378446,28.660933,8724.0,227.478264,227.2455,227.012736,0.02,0.034910,0.037654,0.044204,0.055440,13.224409,46.791217,45.160510,95.771144,0.000088,227.253151,227.268575,19,227.20,0.000088,0.000868,227.24,0.000062,846.6,0.001692,0.000149,4233.0,0.000922,239.835912,243.458192,239.835912,243.458192,0.000869,294.938665,0.098578
73,1720433940000,2024-07-08 10:19:00,227.25,227.25,227.25,227.25,250.0,227.25,3,227.20,13,50.028140,-0.033677,-0.042083,0.008406,79.448622,51.622884,8724.0,227.457761,227.2375,227.017239,0.00,0.017455,0.030124,0.039784,0.052668,12.406482,46.791217,45.160510,68.862275,0.000000,227.253108,227.268520,16,227.20,0.000000,0.000862,227.25,0.000053,765.6,0.001530,0.000142,3828.0,0.001059,236.990368,237.748980,236.990368,237.748980,0.000862,35.530353,0.044176
74,1720434240000,2024-07-08 10:24:00,227.25,227.25,227.25,227.25,315.0,227.25,10,227.21,36,50.028140,-0.028440,-0.039355,0.010914,95.238095,75.355054,8724.0,227.427960,227.2280,227.028040,0.00,0.008727,0.024099,0.035806,0.050035,11.646979,46.791217,45.160510,58.333333,0.000000,227.253067,227.268452,46,227.21,0.000000,0.000669,227.25,0.000101,617.2,0.001233,0.000143,3086.0,0.001326,234.469078,227.250000,234.469078,227.250000,0.000669,-5.672913,0.014764


In [27]:
data_training = combined_data.iloc[1:]
data_training = data_training.fillna(method='ffill').fillna(method='bfill')
data_training.head()

/tmp/ipykernel_27772/2022814271.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_training = data_training.fillna(method='ffill').fillna(method='bfill')


,timestamp,datetime,open,high,low,close,volume,ask_price,ask_size,bid_price,bid_size,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ATR_2,ATR_5,ATR_10,ATR_20,ADX,+DI,-DI,CCI,DLR,TWAP,VWAP,market_liquidity,expected_price,log_return,volatility,mid_price,mean_vol,mean_liq,transaction_cost,5_min_volatility,5_min_volume,5_min_TC,forecast_6Hr_open,forecast_6Hr_high,forecast_6Hr_low,forecast_6Hr_close,forecast_6Hr_volatility,forecast_6Hr_volume,forecast_6Hr_transaction_cost
71,1720433040000,2024-07-08 10:04:00,227.23,227.23,227.23,227.23,233.0,227.23,57,227.17,10,48.455459,-0.047199,-0.045263,-0.001936,24.041585,14.656334,6820.0,227.487690,227.2505,227.013310,0.07,0.049820,0.042068,0.046894,0.057306,14.105253,45.204103,46.50756,136.524823,0.000308,227.253194,227.269230,67,227.17,0.000308,0.000957,227.23,0.000070,573.0,0.001145,0.000124,2865.0,0.000912,252.424985,252.424985,252.424985,252.424985,0.000869,-173.795700,-0.019360
72,1720433520000,2024-07-08 10:12:00,227.23,227.25,227.23,227.25,1904.0,227.25,14,227.20,5,50.028140,-0.039872,-0.044185,0.004313,51.378446,28.660933,8724.0,227.478264,227.2455,227.012736,0.02,0.034910,0.037654,0.044204,0.055440,13.224409,46.791217,45.16051,95.771144,0.000088,227.253151,227.268575,19,227.20,0.000088,0.000868,227.24,0.000062,846.6,0.001692,0.000149,4233.0,0.000922,239.835912,243.458192,239.835912,243.458192,0.000869,294.938665,0.098578
73,1720433940000,2024-07-08 10:19:00,227.25,227.25,227.25,227.25,250.0,227.25,3,227.20,13,50.028140,-0.033677,-0.042083,0.008406,79.448622,51.622884,8724.0,227.457761,227.2375,227.017239,0.00,0.017455,0.030124,0.039784,0.052668,12.406482,46.791217,45.16051,68.862275,0.000000,227.253108,227.268520,16,227.20,0.000000,0.000862,227.25,0.000053,765.6,0.001530,0.000142,3828.0,0.001059,236.990368,237.748980,236.990368,237.748980,0.000862,35.530353,0.044176
74,1720434240000,2024-07-08 10:24:00,227.25,227.25,227.25,227.25,315.0,227.25,10,227.21,36,50.028140,-0.028440,-0.039355,0.010914,95.238095,75.355054,8724.0,227.427960,227.2280,227.028040,0.00,0.008727,0.024099,0.035806,0.050035,11.646979,46.791217,45.16051,58.333333,0.000000,227.253067,227.268452,46,227.21,0.000000,0.000669,227.25,0.000101,617.2,0.001233,0.000143,3086.0,0.001326,234.469078,227.250000,234.469078,227.250000,0.000669,-5.672913,0.014764
75,1720434300000,2024-07-08 10:25:00,227.25,227.25,227.25,227.25,269.0,227.25,8,227.24,7,50.028140,-0.024013,-0.036286,0.012273,100.000000,91.562239,8724.0,227.348809,227.2130,227.077191,0.00,0.004364,0.019279,0.032225,0.047533,10.941726,46.791217,45.16051,55.555556,0.000000,227.253026,227.268394,15,227.24,0.000000,0.000620,227.25,0.000079,594.2,0.001187,0.000144,2971.0,0.001228,226.982810,227.250000,226.982810,227.250000,0.000620,-24.058472,0.015161


In [28]:
data_training.columns

Index(['timestamp', 'datetime', 'open', 'high', 'low', 'close', 'volume',
       'ask_price', 'ask_size', 'bid_price', 'bid_size', 'RSI', 'MACD',
       'MACD_signal', 'MACD_hist', 'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB',
       'Middle_BB', 'Lower_BB', 'ATR_1', 'ATR_2', 'ATR_5', 'ATR_10', 'ATR_20',
       'ADX', '+DI', '-DI', 'CCI', 'DLR', 'TWAP', 'VWAP', 'market_liquidity',
       'expected_price', 'log_return', 'volatility', 'mid_price', 'mean_vol',
       'mean_liq', 'transaction_cost', '5_min_volatility', '5_min_volume',
       '5_min_TC', 'forecast_6Hr_open', 'forecast_6Hr_high',
       'forecast_6Hr_low', 'forecast_6Hr_close', 'forecast_6Hr_volatility',
       'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost'],
      dtype='object')

In [88]:
len(data_training.columns)

51

In [29]:
train_data = data_training.iloc[:8000]
test_data = data_training.iloc[8000:]

In [59]:
train_data.columns

Index(['timestamp', 'datetime', 'open', 'high', 'low', 'close', 'volume',
       'ask_price', 'ask_size', 'bid_price', 'bid_size', 'RSI', 'MACD',
       'MACD_signal', 'MACD_hist', 'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB',
       'Middle_BB', 'Lower_BB', 'ATR_1', 'ATR_2', 'ATR_5', 'ATR_10', 'ATR_20',
       'ADX', '+DI', '-DI', 'CCI', 'DLR', 'TWAP', 'VWAP', 'market_liquidity',
       'expected_price', 'log_return', 'volatility', 'mid_price', 'mean_vol',
       'mean_liq', 'transaction_cost', '5_min_volatility', '5_min_volume',
       '5_min_TC', 'forecast_6Hr_open', 'forecast_6Hr_high',
       'forecast_6Hr_low', 'forecast_6Hr_close', 'forecast_6Hr_volatility',
       'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost'],
      dtype='object')

# Training

## Environment - For Training

In [30]:
import gym
from gym import spaces
import numpy as np
import pandas as pd

class TradingEnvironment(gym.Env):
    metadata = {'render.modes': ['human']}
    
    # Preferred timeframe: The number of steps that the user wants to complete the trade in
    def __init__(self, data, action_space, preferred_timeframe=390, initial_inventory=10, scenario='medium'):
        super(TradingEnvironment, self).__init__()
        self.data = data
        self.current_step = 0
        
        self.preferred_timeframe = preferred_timeframe
        self.initial_inventory = initial_inventory
        self.remaining_inventory = self.initial_inventory
        self.elapsed_time = 0
        self.trades = []
        self.cumulative_reward = 0
        self.scenario = scenario
        
        # Define scenario-specific penalties
        if self.scenario == 'large':
            self.beta = 1
            self.delta = 1
        elif self.scenario == 'medium-large':
            self.beta = 1e2
            self.delta = 1e2
        elif self.scenario == 'medium':
            self.beta = 1e3
            self.delta = 1e3
        elif self.scenario == 'small-medium':
            self.beta = 1e4
            self.delta = 1e4
        elif self.scenario == 'small':
            self.beta = 1e5
            self.delta = 1e5
        else:
            raise ValueError(f"Unknown scenario: {self.scenario}")

        # Extract state columns - have to change according to requirement
        self.state_columns = ['open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'Stoch_k', 'Stoch_d',
                              'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX', '+DI', '-DI', 'CCI', 'transaction_cost',
                              'forecast_6Hr_open','forecast_6Hr_close','forecast_6Hr_high',
                              'forecast_6Hr_low', 'forecast_6Hr_volatility', 'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost']
        
        # Define action space - [%_slice, timing_of_next_slice] * Sequence_Length
        self.action_space = action_space
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(len(self.state_columns),), dtype=np.float32)
        
    
    def _get_state(self):
        market_conditions = self._next_observation()
#         state = np.append(market_conditions, self.remaining_inventory / self.initial_inventory)
        return market_conditions
    
    def _next_observation(self):
        return self.data[self.state_columns].iloc[self.current_step].values
    
    def reset(self):
        print('------------------------------------------------Class resetted------------------------------------------------')
        self.current_step = 0
        self.cumulative_reward = 0
        self.remaining_inventory = self.initial_inventory
        self.elapsed_time = 0
        self.trades = []
        return self._get_state()
    
    def step(self, action):
        # Adding some noise to the actions
        action = self._add_noise_to_action(action)
        print(f'Action taken: {action}')

        size_of_slice = action[0] * self.remaining_inventory
        # Convert size of slice into whole number
        size_of_slice = int(np.ceil(size_of_slice))

        # Scale action[1] to a desired range, e.g., 1 to 10
        timing_of_slice = int(np.ceil(action[1]))
        print(f'timing_of_slice: {timing_of_slice}')
        
        self.elapsed_time += timing_of_slice
        
        if self.elapsed_time >= self.preferred_timeframe:
            size_of_slice = self.remaining_inventory

        execution_price = self._take_action(size_of_slice)

        # Print current and next step for debugging
        print(f'Current step: {self.current_step}')
        self.current_step += timing_of_slice
        self.elapsed_time += timing_of_slice
        print(f'Next step: {self.current_step}')

        # Ensure the index is sequential
        if self.current_step >= len(self.data):
            self.current_step = len(self.data) - 1

        done = self.remaining_inventory <= 0 or self.elapsed_time >= self.preferred_timeframe
        reward = self._calculate_reward(size_of_slice, execution_price, timing_of_slice)
        self.cumulative_reward += reward
        if done:
            print(f'Cumulative Rewards: {self.cumulative_reward}')

        trade_info = {
                'step': self.current_step,
#                 'timestamp': self.data.index[self.current_step],
                'action': action,
                'price': execution_price,
                'shares': size_of_slice,
                'reward': reward,
                'inventory': self.remaining_inventory,
                'time left': self.preferred_timeframe - self.elapsed_time
            }
        self.trades.append(trade_info)

        info = {
            'step': self.current_step,
            'action': action,
            'price': execution_price
        }

        return self._get_state(), reward, done, info

            
    def _add_noise_to_action(self, action):
        # Add noise to the first action (percentage of inventory)
        noise_action_0 = np.random.normal(0, 0.02, size=action[0].shape)  # Small noise for percentage
        action[0] += noise_action_0
        action[0] = np.clip(action[0], self.action_space.low[0], self.action_space.high[0])

        # Add noise to the second action (timing of next slice)
        noise_action_1 = np.random.normal(0, 5, size=action[1].shape)  # Setting SD to be 5% of the range (1-100)
        action[1] += noise_action_1
        action[1] = np.clip(action[1], self.action_space.low[1], self.action_space.high[1])

        return action

    
    def _take_action(self, size_of_slice):
        self.remaining_inventory -= size_of_slice
        print(f'Remaining inventory: {self.remaining_inventory}')
        if self.remaining_inventory < 0:
            self.remaining_inventory = 0
        execution_price = self.data['close'].iloc[self.current_step]
        return execution_price
    
    def _calculate_transaction_cost(self, volume, daily_volume, volatility=None):
        if volatility is None:
            volatility = self.data['volatility'].iloc[self.current_step]
        return volatility * np.sqrt(volume / daily_volume)
            
    def _calculate_reward(self, size_of_slice, execution_price, timing_of_slice):
        # Constants
        kappa = 0.1

        expected_price = self.data['expected_price'].iloc[self.current_step]
        actual_price = execution_price
        order_size = size_of_slice
        market_liquidity = self.data['market_liquidity'].iloc[self.current_step]
        time_remaining = self.preferred_timeframe - self.elapsed_time
        total_time = self.preferred_timeframe

        # Calculating various components of the reward function
        slippage = expected_price - actual_price
        transaction_costs = self.data['transaction_cost'].iloc[self.current_step]
        
        # Penalize actions taken early in the timeframe (encourage spreading actions)
        early_action_penalty = self.delta * (time_remaining / total_time) ** 2  # Quadratic scaling
        
        if self.scenario in ['small', 'small-medium']:
            small_timestep_penalty = 0 if timing_of_slice > 40 else 100
        elif self.scenario in ['medium', 'medium-large']:
            small_timestep_penalty = 0 if timing_of_slice > 20 else 50
        elif self.scenario == 'large':
            small_timestep_penalty = 0 if timing_of_slice > 10 else 10
        else:
            small_timestep_penalty = 0

        # Combining all the components to calculate the reward
        penalty = (slippage + transaction_costs + early_action_penalty + small_timestep_penalty)
        
        # Adding utility theory in rewards
        reward = -penalty - (2 * kappa * (penalty ** 2))
        
        print(f"Slippage: {slippage} TC: {transaction_costs} Rapid: {early_action_penalty} Time: {small_timestep_penalty}")

        return reward

    
    def render(self, mode='human', close=False):
        print('--------------------------------------------------')
        print(f'Steps: {self.current_step}')
        print(f'Remaining inventory: {self.remaining_inventory}')
        print(f'Cumulative reward: {self.cumulative_reward}')
        self.print_trades()

    def print_trades(self):
        trades_df = pd.DataFrame(self.trades)
        for trade in self.trades:
            print(f"Step: {trade['step']}, Action: {trade['action']}, Price: {trade['price']}, Shares: {trade['shares']}, Reward: {trade['reward']}, Inventory: {trade['inventory']}, TimeLeft: {trade['time left']}")
        
        return self.trades


## Transformer Initialization - U-Net Architecture

### Class Summary

**`UNetTransformerEncoder`**

This class implements a hybrid architecture combining a U-Net style encoder-decoder with a Transformer encoder. It is designed to handle different scenarios (e.g., small, medium, large) by adjusting the parameters of the Transformer layers dynamically.

### Key Components

- **U-Net Layers:**
  - **Encoders (`encoder1`, `encoder2`, `encoder3`, `encoder4`)**: Sequential layers that reduce the input dimensionality while capturing hierarchical features.
  - **Bottleneck:** A central layer that captures the most abstract representation before upsampling.
  - **Decoders (`decoder4`, `decoder3`, `decoder2`, `decoder1`)**: Layers that upsample the feature maps back to the original resolution while combining features from corresponding encoder layers.

- **Transformer Layers:**
  - **Scenario-based Transformer Encoder Layer (`encoder_layer`)**: The Transformer encoder layer is configured based on the specified scenario. Different scenarios adjust the number of heads and dropout rates in the Transformer.
  - **Transformer Encoder (`transformer_encoder`)**: Applies multiple Transformer encoder layers to the output of the U-Net decoders, allowing the model to capture long-range dependencies in the feature maps.

### Scenarios

- **small:** 4 heads, 0.2 dropout
- **small-medium:** 8 heads, 0.15 dropout
- **medium:** 8 heads, 0.1 dropout
- **medium-large:** 16 heads, 0.08 dropout
- **large:** 32 heads, 0.05 dropout

### Parameters

- **`in_channels (int)`**: Number of input channels for the U-Net.
- **`out_channels (int)`**: Number of output channels for the U-Net.
- **`scenario (str)`**: Scenario to configure the Transformer layers (default is 'medium').

### Methods

- **`_block(self, in_channels, out_channels)`**: Defines a convolutional block with Conv1D, BatchNorm, and ReLU activation.
- **`_get_encoder_layer(self, features_dim, scenario)`**: Configures the Transformer encoder layer based on the scenario.
- **`forward(self, x)`**: Passes the input through the U-Net encoder-decoder architecture followed by the Transformer encoder.


In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3 import PPO
from stable_baselines3.common.policies import ActorCriticPolicy
from gym import spaces


class UNetTransformerEncoder(nn.Module):
    def __init__(self, in_channels, out_channels, scenario='medium'):
        super(UNetTransformerEncoder, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.scenario = scenario

        # Define U-Net layers
        self.encoder1 = self._block(in_channels, 64)
        self.encoder2 = self._block(64, 128)
        self.encoder3 = self._block(128, 256)
        self.encoder4 = self._block(256, 512)
        self.bottleneck = self._block(512, 1024)
        self.decoder4 = self._block(1024 + 512, 512)
        self.decoder3 = self._block(512 + 256, 256)
        self.decoder2 = self._block(256 + 128, 128)
        self.decoder1 = self._block(128 + 64, out_channels)

        # Define Transformer layers based on scenario
        self.encoder_layer = self._get_encoder_layer(out_channels, scenario)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=6)

    def _block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    # Dynamic weight change based on scenario
    def _get_encoder_layer(self, features_dim, scenario):
        if scenario == 'small':
            return nn.TransformerEncoderLayer(d_model=features_dim, nhead=4, dropout=0.2)
        elif scenario == 'small-medium':
            return nn.TransformerEncoderLayer(d_model=features_dim, nhead=8, dropout=0.15)
        elif scenario == 'medium':
            return nn.TransformerEncoderLayer(d_model=features_dim, nhead=8, dropout=0.1)
        elif scenario == 'medium-large':
            return nn.TransformerEncoderLayer(d_model=features_dim, nhead=16, dropout=0.08)
        elif scenario == 'large':
            return nn.TransformerEncoderLayer(d_model=features_dim, nhead=32, dropout=0.05)
        else:
            raise ValueError(f"Unknown scenario: {scenario}")

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(F.max_pool1d(enc1, 2))
        enc3 = self.encoder3(F.max_pool1d(enc2, 2))
        enc4 = self.encoder4(F.max_pool1d(enc3, 2))

        bottleneck = self.bottleneck(F.max_pool1d(enc4, 2))

        dec4 = self.decoder4(torch.cat((F.interpolate(bottleneck, scale_factor=2), enc4), dim=1))
        dec3 = self.decoder3(torch.cat((F.interpolate(dec4, scale_factor=2), enc3), dim=1))
        dec2 = self.decoder2(torch.cat((F.interpolate(dec3, scale_factor=2), enc2), dim=1))
        dec1 = self.decoder1(torch.cat((F.interpolate(dec2, scale_factor=2), enc1), dim=1))

        # Transformer encoding
        x = self.transformer_encoder(dec1.permute(2, 0, 1)).permute(1, 2, 0)  # Permute for transformer encoder and back

        return x


In [32]:
class CustomUNetTransformerModel(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Box, features_dim: int = 256, scenario: str = 'medium'):
        super(CustomUNetTransformerModel, self).__init__(observation_space, features_dim)
        self.embedding = nn.Linear(observation_space.shape[0], features_dim)  # Adapt the input size if necessary
        self.scenario = scenario
        self.unet_transformer = UNetTransformerEncoder(1, features_dim, scenario)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = self.embedding(observations)
        x = x.unsqueeze(1)
        x = self.unet_transformer(x)
        x = x.mean(dim=2)
        return x

In [33]:
from stable_baselines3.common.policies import ActorCriticPolicy
from gym import spaces

class CustomTransformerPolicy(ActorCriticPolicy):
    def __init__(self, observation_space, action_space, lr_schedule, scenario='medium', *args, **kwargs):
        super(CustomTransformerPolicy, self).__init__(observation_space, action_space, lr_schedule, 
                                                      features_extractor_class=CustomUNetTransformerModel, 
                                                      features_extractor_kwargs={'features_dim': 256, 'scenario': scenario},
                                                      *args, **kwargs)

## Meta-Learner for Dynamic Scenario detection

In [34]:
class MetaLearner:
    def __init__(self):
        # Define thresholds for different buckets
        self.small_threshold = 10
        self.small_medium_threshold = 100
        self.medium_threshold = 500
        self.medium_large_threshold = 2000
        self.large_threshold = 10000

    def classify_scenario(self, transaction_size, timeframe=390):
        """
        Classify the scenario based on the transaction size.
        The default timeframe is set to 390 minutes (1 trading day).
        """
        if transaction_size < self.small_threshold:
            return 'small'
        elif self.small_threshold <= transaction_size < self.small_medium_threshold:
            return 'small-medium'
        elif self.small_medium_threshold <= transaction_size < self.medium_threshold:
            return 'medium'
        elif self.medium_threshold <= transaction_size < self.medium_large_threshold:
            return 'medium-large'
        elif self.medium_large_threshold <= transaction_size < self.large_threshold:
            return 'large'
        else:
            return 'large'

## Models

We are using 5 base models to start with:
To begin with we can assume time frame is 1 day. We can then break the order down by size so <10 shares is one bucket (small), 10 - 100 shares is (small medium), 100 - 500 (medium), 500 - 2k (medium large), and 2k - 10k (large)

### 1. Small (<10 shares)

In [ ]:
import torch
from stable_baselines3 import PPO

# Define transaction size and timeframe
transaction_size = 9

# Meta-Learner instance and scenario classification
meta = MetaLearner()
scenario = meta.classify_scenario(transaction_size)
print(f"Classified Scenario: {scenario}")

# Define the best hyperparameters
best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                        'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}

# Define action space for small model
action_space = spaces.Box(low=np.array([0.33, 30]), high=np.array([1, 50]), dtype=np.float32)

# Create the trading environment
env_small = TradingEnvironment(train_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)

model_small = PPO(CustomTransformerPolicy, env_small, verbose=1, policy_kwargs={'scenario': scenario}, **best_hyperparameters)

# Train the model
model_small.learn(total_timesteps=10000)



In [36]:
# Evaluate the model
new_env = TradingEnvironment(test_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)
obs = new_env.reset()

for _ in range(len(test_data)):
    action, _states = model_small.predict(obs)
    obs, rewards, done, info = new_env.step(action)
    if done:
        break

# Render the final state
new_env.render()

------------------------------------------------Class resetted------------------------------------------------
Action taken: [ 0.3460971 30.       ]
timing_of_slice: 30
Remaining inventory: 5
Current step: 0
Next step: 30
Slippage: 0.1599999999999966 TC: 0.21752093650283663 Rapid: 71597.63313609468 Time: 100
Action taken: [ 0.95473987 36.531536  ]
timing_of_slice: 37
Remaining inventory: 0
Current step: 30
Next step: 67
Slippage: -0.009999999999990905 TC: 0.10389182275419714 Rapid: 43087.44247205786 Time: 100
Cumulative Rewards: -1401268491.4184744
--------------------------------------------------
Steps: 67
Remaining inventory: 0
Cumulative reward: -1401268491.4184744
Step: 30, Action: [ 0.3460971 30.       ], Price: 225.78, Shares: 4, Reward: -1028192644.4458084, Inventory: 5, TimeLeft: 330
Step: 67, Action: [ 0.95473987 36.531536  ], Price: 224.01, Shares: 5, Reward: -373075846.972666, Inventory: 0, TimeLeft: 256


In [37]:
import torch
import h5py

def save_model(model, filepath):
    # Save the model parameters
    torch.save(model.policy.state_dict(), filepath)
    print(f"Model saved to {filepath}")

# Example usage
save_model(model_small, 'Models/model_small.h5')

Model saved to Models/model_small.h5


### 2. Small-Medium

In [ ]:
import torch
from stable_baselines3 import PPO

# Define transaction size and timeframe
transaction_size = 70

# Meta-Learner instance and scenario classification
meta = MetaLearner()
scenario = meta.classify_scenario(transaction_size)
print(f"Classified Scenario: {scenario}")

# Define the best hyperparameters
best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                        'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}

# Define action space for small model
action_space = spaces.Box(low=np.array([0.2, 20]), high=np.array([0.66, 30]), dtype=np.float32)

# Create the trading environment
env_small_med = TradingEnvironment(train_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)

model_small_med = PPO(CustomTransformerPolicy, env_small_med, verbose=1, **best_hyperparameters)

# Train the model
model_small_med.learn(total_timesteps=1000)

# Evaluate the model
new_env = TradingEnvironment(test_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)
obs = new_env.reset()

for _ in range(len(test_data)):
    action, _states = model_small_med.predict(obs)
    obs, rewards, done, info = new_env.step(action)
    if done:
        break

# Render the final state
new_env.render()

In [39]:
import torch
import h5py

def save_model(model, filepath):
    # Save the model parameters
    torch.save(model.policy.state_dict(), filepath)
    print(f"Model saved to {filepath}")

# Example usage
save_model(model_small_med, 'Models/model_small_med.h5')

Model saved to Models/model_small_med.h5


### 3. Medium

In [40]:
len(train_data.iloc[0])

51

In [ ]:
import torch
from stable_baselines3 import PPO

# Define transaction size and timeframe
transaction_size = 300

# Meta-Learner instance and scenario classification
meta = MetaLearner()
scenario = meta.classify_scenario(transaction_size)
print(f"Classified Scenario: {scenario}")

# Define the best hyperparameters
best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                        'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}

# Define action space for small model
action_space = spaces.Box(low=np.array([0.20, 30]), high=np.array([0.50, 50]), dtype=np.float32)

# Create the trading environment
env_med = TradingEnvironment(train_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)

model_med = PPO(CustomTransformerPolicy, env_med, verbose=1, policy_kwargs={'scenario': scenario}, **best_hyperparameters)

# Train the model
model_med.learn(total_timesteps=10000)

# Evaluate the model
new_env = TradingEnvironment(test_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)
obs = new_env.reset()

for _ in range(len(test_data)):
    action, _states = model_med.predict(obs)
    obs, rewards, done, info = new_env.step(action)
    if done:
        break

# Render the final state
new_env.render()

In [42]:
import torch
import h5py

def save_model(model, filepath):
    # Save the model parameters
    torch.save(model.policy.state_dict(), filepath)
    print(f"Model saved to {filepath}")

# Example usage
save_model(model_med, 'Models/model_med.h5')

Model saved to Models/model_med.h5


### 4. Medium - Large

In [ ]:
import torch
from stable_baselines3 import PPO

# Define transaction size and timeframe
transaction_size = 1500

# Meta-Learner instance and scenario classification
meta = MetaLearner()
scenario = meta.classify_scenario(transaction_size)
print(f"Classified Scenario: {scenario}")

# Define the best hyperparameters
best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                        'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}

# Define action space for small model
action_space = spaces.Box(low=np.array([0.10, 30]), high=np.array([0.40, 50]), dtype=np.float32)

# Create the trading environment
env_med_lg = TradingEnvironment(train_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)

model_med_lg = PPO(CustomTransformerPolicy, env_med_lg, verbose=1, policy_kwargs={'scenario': scenario}, **best_hyperparameters)

# Train the model
model_med_lg.learn(total_timesteps=10000)

# Evaluate the model
new_env = TradingEnvironment(test_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)
obs = new_env.reset()

for _ in range(len(test_data)):
    action, _states = model_med_lg.predict(obs)
    obs, rewards, done, info = new_env.step(action)
    if done:
        break

# Render the final state
new_env.render()

In [44]:
import torch
import h5py

def save_model(model, filepath):
    # Save the model parameters
    torch.save(model.policy.state_dict(), filepath)
    print(f"Model saved to {filepath}")

# Example usage
save_model(model_med_lg, 'Models/model_med_lg.h5')

Model saved to Models/model_med_lg.h5


### 5. Large

In [ ]:
import torch
from stable_baselines3 import PPO

# Define transaction size and timeframe
transaction_size = 10000

# Meta-Learner instance and scenario classification
meta = MetaLearner()
scenario = meta.classify_scenario(transaction_size)
print(f"Classified Scenario: {scenario}")

# Define the best hyperparameters
best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                        'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}

# Define action space for small model
action_space = spaces.Box(low=np.array([0.05, 30]), high=np.array([0.33, 50]), dtype=np.float32)

# Create the trading environment
env_lg = TradingEnvironment(train_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)

model_lg = PPO(CustomTransformerPolicy, env_lg, verbose=1,policy_kwargs={'scenario': scenario}, **best_hyperparameters)

# Train the model
model_lg.learn(total_timesteps=1000)



In [46]:
# Evaluate the model
new_env = TradingEnvironment(test_data, scenario=scenario, action_space=action_space, initial_inventory=transaction_size)
obs = new_env.reset()

for _ in range(len(test_data)):
    action, _states = model_lg.predict(obs)
    obs, rewards, done, info = new_env.step(action)
    if done:
        break

# Render the final state
new_env.render()

------------------------------------------------Class resetted------------------------------------------------
Action taken: [ 0.06754091 34.404556  ]
timing_of_slice: 35
Remaining inventory: 9324
Current step: 0
Next step: 35
Slippage: 0.1599999999999966 TC: 0.19921079092218094 Rapid: 0.6732412886259039 Time: 0
Action taken: [ 0.3037092 30.       ]
timing_of_slice: 30
Remaining inventory: 6492
Current step: 35
Next step: 65
Slippage: 0.05000000000001137 TC: 0.10943715058313846 Rapid: 0.4444444444444444 Time: 0
Action taken: [ 0.05 30.  ]
timing_of_slice: 30
Remaining inventory: 6167
Current step: 65
Next step: 95
Slippage: 0.15000000000000568 TC: 0.08847346663141517 Rapid: 0.2629848783694937 Time: 0
Action taken: [ 0.06161678 31.08051   ]
timing_of_slice: 32
Remaining inventory: 5787
Current step: 95
Next step: 127
Slippage: 0.3001000000000147 TC: 0.10659876729455547 Rapid: 0.1216042077580539 Time: 0
Action taken: [ 0.09320737 30.        ]
timing_of_slice: 30
Remaining inventory: 5247

In [47]:
import torch
import h5py

def save_model(model, filepath):
    # Save the model parameters
    torch.save(model.policy.state_dict(), filepath)
    print(f"Model saved to {filepath}")

# Example usage
save_model(model_lg, 'Models/model_lg.h5')

Model saved to Models/model_lg.h5


# Testing

# Code for getting data ready for inferencing

### Assumptions:
1. Input Data is just pulled data and has only OHLCV values
2. There are no features/forecasts in the data row - Add that
3. Just pulled data features: [O,H,L,C,V]
4. The below code just pulls OHLCV but can integrate new code to add bid ask as well

### 1. Pull Data

In [49]:
import sys
sys.path.append('Data_Script')
from Data_Script.fetch_merge_data import fetch_and_merge_data

df = fetch_and_merge_data('AAPL','2024-07-07','2024-08-07')

df.head()

Total Quote Process Time:  97.71320533752441  seconds.
Merged data saved to merged_data_AAPL_2024-07-07_2024-08-07.csv


/home/ec2-user/SageMaker/Data_Script/fetch_merge_data.py:107: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  quotes = df.resample('min').agg({


,timestamp,datetime,open,high,low,close,volume,ask_price,ask_size,bid_price,bid_size
0,1720425600000,2024-07-08 08:00:00,227.60,227.6,226.61,226.82,3947.0,0.0,0,0.0,0
1,1720425660000,2024-07-08 08:01:00,226.90,226.9,226.90,226.90,1914.0,0.0,0,0.0,0
2,1720425720000,2024-07-08 08:02:00,226.87,227.0,226.87,227.00,1325.0,0.0,0,0.0,0
3,1720425780000,2024-07-08 08:03:00,227.00,227.1,227.00,227.10,1037.0,0.0,0,0.0,0
4,1720425840000,2024-07-08 08:04:00,227.20,227.2,227.20,227.20,1819.0,0.0,0,0.0,0


In [77]:
data_testing = df.copy()

### 2. Add Tech Indicators + TC

In [ ]:
# Install TA-Lib using conda
!conda install -c conda-forge ta-lib -y

In [78]:
# Define Tech Indicators
import pandas as pd
import numpy as np
import talib as ta
import numpy as np
import math

class TechnicalIndicators:
    def __init__(self, data):
        """
        Initializes the TechnicalIndicators class with the provided data.

        Parameters:
        - data: A pandas DataFrame containing financial data with columns like 'close', 'high', 'low', 'volume', etc.
        """
        self.data = data

    def add_momentum_indicators(self):
        """
        Adds momentum indicators such as RSI, MACD, and Stochastic Oscillator to the data.
        """
        # Relative Strength Index (RSI)
        self.data['RSI'] = ta.RSI(self.data['close'], timeperiod=14)
        
        # Moving Average Convergence Divergence (MACD)
        self.data['MACD'], self.data['MACD_signal'], self.data['MACD_hist'] = ta.MACD(
            self.data['close'], fastperiod=12, slowperiod=26, signalperiod=9)
        
        # Stochastic Oscillator
        self.data['Stoch_k'], self.data['Stoch_d'] = ta.STOCH(
            self.data['high'], self.data['low'], self.data['close'], fastk_period=14, slowk_period=3, slowd_period=3)

    def add_volume_indicators(self):
        """
        Adds volume indicators such as On-Balance Volume (OBV) to the data.
        """
        # On-Balance Volume (OBV)
        self.data['OBV'] = ta.OBV(self.data['close'], self.data['volume'])
        
    def add_TC(self):
        window_size = 5
        self.data['mid_price'] = (self.data['high'] + self.data['low']) / 2
        self.data["mean_vol"] = self.data['mid_price'].pct_change().rolling(window=window_size).mean()
        self.data["mean_liq"] = self.data['volume'].rolling(window=window_size).mean()
        self.data = self.data.iloc[35:,:]


        # AC Calculation -> 
        x0 = 5000  # Initial number of shares to trade
        T = 1.0    # Total time horizon (e.g., 1 day)
        N = min(x0, 2400)  # Number of discrete time intervals
        eta = 0.0000001    # Temporary/permanent impact coefficient
        sigma = 0.02       # Volatility of the asset
        lambda_ = 0.1      # Risk aversion parameter

        # Time interval
        dt = 1

        # Cost function for the Almgren-Chriss model
        def cost_function(x, eta, sigma, x0=10):
            x_cumsum = np.cumsum(x)
            x_half = x / 2

            temp_cost = np.sum(eta * (x**2) / dt)
            perm_cost = np.sum(eta * x * (x0 - x_cumsum + x_half))
            var_cost = np.sum(lambda_ * sigma**2 * (x**2) * dt)

            return (1 / (temp_cost + perm_cost + var_cost) / x0) * 10**(math.log10(x0) * 4 - 6)

        def almgren(row):
            x0 = 5000
            N = min(x0, 2400)

            eta = 1 / row['mean_liq'] if row['mean_liq'] != 0 else 0.0000001
            sigma = row['mean_vol']

            # Initial guess: trade evenly across all intervals
            x_init = np.ones(N) * (x0 / N)

            return cost_function(x_init, eta, sigma, x0) / x0
        
        self.data['transaction_cost'] = self.data.apply(almgren, axis=1)

    def add_volatility_indicators(self):
        """
        Adds volatility indicators such as Bollinger Bands and Average True Range (ATR) to the data.
        """
        # Bollinger Bands
        self.data['Upper_BB'], self.data['Middle_BB'], self.data['Lower_BB'] = ta.BBANDS(self.data['close'], timeperiod=20)
        
        # Average True Range (ATR) for different periods
        self.data['ATR_1'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=1)
        self.data['ATR_2'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=2)
        self.data['ATR_5'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=5)
        self.data['ATR_10'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=10)
        self.data['ATR_20'] = ta.ATR(self.data['high'], self.data['low'], self.data['close'], timeperiod=20)
        
    def add_volatility(self, window=15):
        """
        Adds dynamic volatility calculation using log returns and a rolling window.

        Parameters:
        - window: The rolling window size for calculating volatility (default is 15).
        """
        # Log returns
        self.data['log_return'] = np.log(self.data['close'] / self.data['close'].shift(1))
        
        # Rolling volatility
        self.data['volatility'] = self.data['log_return'].rolling(window=window).std() * np.sqrt(window)

    def add_trend_indicators(self):
        """
        Adds trend indicators such as ADX, +DI, -DI, and CCI to the data.
        """
        # Average Directional Index (ADX)
        self.data['ADX'] = ta.ADX(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        
        # Plus Directional Indicator (+DI)
        self.data['+DI'] = ta.PLUS_DI(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        
        # Minus Directional Indicator (-DI)
        self.data['-DI'] = ta.MINUS_DI(self.data['high'], self.data['low'], self.data['close'], timeperiod=14)
        
        # Commodity Channel Index (CCI)
        self.data['CCI'] = ta.CCI(self.data['high'], self.data['low'], self.data['close'], timeperiod=5)
        
    def add_5_min_indicators(self):
        # code to add 5 mins volume, volatility, TC
        self.data['5_min_volatility'] = self.data['volatility'].transform(lambda x: x.rolling(window=5).std())
        self.data['5_min_volume'] = self.data['volume'].transform(lambda x: x.rolling(window=5).sum())
        self.data['5_min_TC'] = self.data['transaction_cost'].shift(5)

    def add_other_indicators(self):
        """
        Adds other indicators such as DLR, TWAP, VWAP, market liquidity, and expected price to the data.
        """
        # Daily Log Returns (DLR)
        self.data['DLR'] = np.log(self.data['close'] / self.data['close'].shift(1))
        
        # Time-Weighted Average Price (TWAP)
        self.data['TWAP'] = self.data['close'].expanding().mean()
        
        # Volume-Weighted Average Price (VWAP)
        self.data['VWAP'] = (self.data['volume'] * (self.data['high'] + self.data['low']) / 2).cumsum() / self.data['volume'].cumsum()
        
        # Market Liquidity
        self.data['market_liquidity'] = self.data['bid_size'] + self.data['ask_size']
        
        # Expected Price
        self.data['expected_price'] = self.data['ask_price']
        

    def add_all_indicators(self):
        """
        Adds all the defined indicators to the data.
        
        Returns:
        - The updated DataFrame with all indicators added.
        """
        self.add_momentum_indicators()
        self.add_volume_indicators()
        self.add_volatility_indicators()
        self.add_trend_indicators()
        self.add_other_indicators()
        self.add_volatility()
        self.add_TC()
        self.add_5_min_indicators()
        return self.data

In [79]:
# Create an instance of TechnicalIndicators
indicators = TechnicalIndicators(df)

# Add all indicators
data_with_indicators = indicators.add_all_indicators()

# Discard NaN values 
data_testing = data_with_indicators.iloc[35:]

# Display the updated data
data_testing.head(10)

/tmp/ipykernel_27772/4236684154.py:82: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['transaction_cost'] = self.data.apply(almgren, axis=1)
/tmp/ipykernel_27772/4236684154.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['5_min_volatility'] = self.data['volatility'].transform(lambda x: x.rolling(window=5).std())
/tmp/ipykernel_27772/4236684154.py:130: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

,timestamp,datetime,open,high,low,close,volume,ask_price,ask_size,bid_price,bid_size,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ATR_2,ATR_5,ATR_10,ATR_20,ADX,+DI,-DI,CCI,DLR,TWAP,VWAP,market_liquidity,expected_price,log_return,volatility,mid_price,mean_vol,mean_liq,transaction_cost,5_min_volatility,5_min_volume,5_min_TC
70,1720432740000,2024-07-08 09:59:00,227.16,227.16,227.16,227.16,384.0,227.21,17,227.13,12,42.582713,-0.053827,-0.044779,-0.009049,10.562766,9.623502,6587.0,227.503063,227.2585,227.013937,0.02,0.029639,0.035085,0.044326,0.056637,15.080945,39.321896,51.500034,-10.752688,0.000088,227.253521,227.269340,29,227.21,0.000088,0.001054,227.16,-0.000009,614.4,0.001228,0.000079,3072.0,0.001077
71,1720433040000,2024-07-08 10:04:00,227.23,227.23,227.23,227.23,233.0,227.23,57,227.17,10,48.455459,-0.047199,-0.045263,-0.001936,24.041585,14.656334,6820.0,227.487690,227.2505,227.013310,0.07,0.049820,0.042068,0.046894,0.057306,14.105253,45.204103,46.507560,136.524823,0.000308,227.253194,227.269230,67,227.23,0.000308,0.000957,227.23,0.000070,573.0,0.001145,0.000124,2865.0,0.000912
72,1720433520000,2024-07-08 10:12:00,227.23,227.25,227.23,227.25,1904.0,227.25,14,227.20,5,50.028140,-0.039872,-0.044185,0.004313,51.378446,28.660933,8724.0,227.478264,227.2455,227.012736,0.02,0.034910,0.037654,0.044204,0.055440,13.224409,46.791217,45.160510,95.771144,0.000088,227.253151,227.268575,19,227.25,0.000088,0.000868,227.24,0.000062,846.6,0.001692,0.000149,4233.0,0.000922
73,1720433940000,2024-07-08 10:19:00,227.25,227.25,227.25,227.25,250.0,227.25,3,227.20,13,50.028140,-0.033677,-0.042083,0.008406,79.448622,51.622884,8724.0,227.457761,227.2375,227.017239,0.00,0.017455,0.030124,0.039784,0.052668,12.406482,46.791217,45.160510,68.862275,0.000000,227.253108,227.268520,16,227.25,0.000000,0.000862,227.25,0.000053,765.6,0.001530,0.000142,3828.0,0.001059
74,1720434240000,2024-07-08 10:24:00,227.25,227.25,227.25,227.25,315.0,227.25,10,227.21,36,50.028140,-0.028440,-0.039355,0.010914,95.238095,75.355054,8724.0,227.427960,227.2280,227.028040,0.00,0.008727,0.024099,0.035806,0.050035,11.646979,46.791217,45.160510,58.333333,0.000000,227.253067,227.268452,46,227.25,0.000000,0.000669,227.25,0.000101,617.2,0.001233,0.000143,3086.0,0.001326
75,1720434300000,2024-07-08 10:25:00,227.25,227.25,227.25,227.25,269.0,227.25,8,227.24,7,50.028140,-0.024013,-0.036286,0.012273,100.000000,91.562239,8724.0,227.348809,227.2130,227.077191,0.00,0.004364,0.019279,0.032225,0.047533,10.941726,46.791217,45.160510,55.555556,0.000000,227.253026,227.268394,15,227.25,0.000000,0.000620,227.25,0.000079,594.2,0.001187,0.000144,2971.0,0.001228
76,1720434360000,2024-07-08 10:26:00,227.25,227.25,227.25,227.25,318.0,227.25,0,227.24,0,50.028140,-0.020270,-0.033083,0.012813,100.000000,98.412698,8724.0,227.312733,227.2055,227.098267,0.00,0.002182,0.015423,0.029003,0.045156,10.286849,46.791217,45.160510,41.666667,0.000000,227.252987,227.268326,0,227.25,0.000000,0.000620,227.25,0.000018,611.2,0.001221,0.000127,3056.0,0.001145
77,1720434420000,2024-07-08 10:27:00,227.25,227.25,227.25,227.25,255.0,227.25,3,227.24,2,50.028140,-0.017107,-0.029888,0.012780,100.000000,100.000000,8724.0,227.303419,227.2030,227.102581,0.00,0.001091,0.012339,0.026102,0.042899,9.678748,46.791217,45.160510,0.000000,0.000000,227.252949,227.268272,5,227.25,0.000000,0.000620,227.25,0.000009,281.4,0.000562,0.000105,1407.0,0.001692
78,1720434540000,2024-07-08 10:29:00,227.26,227.26,227.26,227.26,491.0,227.33,23,227.24,11,51.189716,-0.013637,-0.026638,0.013001,100.000000,100.000000,9215.0,227.289219,227.2000,227.110781,0.01,0.005545,0.011871,0.024492,0.041254,9.282327,47.966709,44.162821,166.666667,0.000044,227.253038,227.268226,34,227.33,0.000044,0.000615,227.26,0.000009,329.6,0.000659,0.000022,1648.0,0.001530
79,1720434780000,2024-07-08 10:33:00,227.24,227.24,227.24,227.24,549.0,227.24,143,227.21,18,48.749080,-0.012357,-0.023782,0.011424,94.871795,98.290598,8666.0,227.292907,227.2020,227

Now we have the input data setup with Tech Indicators and TC values (using AC)

### 3. Add forecasts for input data

In [83]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import ARIMA, ExponentialSmoothing
from joblib import Parallel, delayed
import warnings

def forecast_last_row(data, forecast_steps, columns, window_size):
    last_idx = len(data) - 1
    row_forecasts = {}
    for indicator, column in columns.items():
        steps, freq = forecast_steps[indicator]
        start_idx = max(0, last_idx - window_size)
        series = data[column].iloc[start_idx:last_idx+1]
        if len(series) < 2:
            row_forecasts[f'forecast_{indicator}'] = None
            continue
        if indicator in ['open', 'high', 'low', 'close', 'transaction_cost']:
            model = ExponentialSmoothing(series, trend='add', seasonal=None)
        elif indicator == 'volatility':
            model = ARIMA(series, order=(5, 1, 0))
        elif indicator == 'volume':
            shift = 1 if series.min() <= 0 else 0
            transformed_series = np.log(series + shift + 1)
            model = ExponentialSmoothing(transformed_series, trend='add', seasonal=None)
        model_fit = model.fit()
        forecast_values = model_fit.forecast(steps=steps)
        if indicator == 'volume':
            forecast_values = np.exp(forecast_values) - 1 - shift
            forecast_values[forecast_values < 0] = 0  # Ensure non-negative values
        row_forecasts[f'forecast_6Hr_{indicator}'] = forecast_values.iloc[-1]
    return pd.Series(row_forecasts)

# data_testing['datetime'] = pd.to_datetime(data_testing['datetime'], unit='ms')
# data_testing.set_index('datetime', inplace=True)

print(data_testing.head())
forecast_steps = {
    'open': (360, '1T'),
    'high': (360, '1T'),
    'low': (360, '1T'),
    'close': (360, '1T'),
    'volatility': (360, '1T'),
    'volume': (360, '1T'),
    'transaction_cost': (360, '1T')
}
columns = {col: col for col in forecast_steps}
last_row_forecast = forecast_last_row(data_testing, forecast_steps, columns, 300)
print(last_row_forecast)

        timestamp             datetime    open    high     low   close  \
70  1720432740000  2024-07-08 09:59:00  227.16  227.16  227.16  227.16   
71  1720433040000  2024-07-08 10:04:00  227.23  227.23  227.23  227.23   
72  1720433520000  2024-07-08 10:12:00  227.23  227.25  227.23  227.25   
73  1720433940000  2024-07-08 10:19:00  227.25  227.25  227.25  227.25   
74  1720434240000  2024-07-08 10:24:00  227.25  227.25  227.25  227.25   

    volume  ask_price  ask_size  bid_price  bid_size        RSI      MACD  \
70   384.0     227.21        17     227.13        12  42.582713 -0.053827   
71   233.0     227.23        57     227.17        10  48.455459 -0.047199   
72  1904.0     227.25        14     227.20         5  50.028140 -0.039872   
73   250.0     227.25         3     227.20        13  50.028140 -0.033677   
74   315.0     227.25        10     227.21        36  50.028140 -0.028440   

    MACD_signal  MACD_hist    Stoch_k    Stoch_d     OBV    Upper_BB  \
70    -0.044779  -0.

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [84]:
import pandas as pd

# Assuming data is your DataFrame and last_row_forecast is a Series

# Get the last row without the index
last_row_df = data_testing.iloc[-1].to_frame().T.reset_index(drop=True)

# Convert the last_row_forecast to a DataFrame and reset its index
last_row_forecast_df = last_row_forecast.to_frame().T.reset_index(drop=True)

# Combine them horizontally
input_row = pd.concat([last_row_df, last_row_forecast_df], axis=1)

# View the combined input row
input_row

,timestamp,datetime,open,high,low,close,volume,ask_price,ask_size,bid_price,bid_size,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ATR_2,ATR_5,ATR_10,ATR_20,ADX,+DI,-DI,CCI,DLR,TWAP,VWAP,market_liquidity,expected_price,log_return,volatility,mid_price,mean_vol,mean_liq,transaction_cost,5_min_volatility,5_min_volume,5_min_TC,forecast_6Hr_open,forecast_6Hr_high,forecast_6Hr_low,forecast_6Hr_close,forecast_6Hr_volatility,forecast_6Hr_volume,forecast_6Hr_transaction_cost
0,1723061520000,2024-08-07 20:12:00,209.6,209.74,209.6,209.74,8933.0,209.74,232,209.71,528,45.983343,-0.169979,-0.191083,0.021104,53.051643,52.756438,12418709.0,210.152319,209.7375,209.322681,0.17,0.125099,0.171488,0.229091,0.267226,28.23448,15.057196,27.026711,58.882236,0.000811,222.796691,221.889634,760,209.74,0.000811,0.002942,209.67,0.000143,7155.0,0.014298,0.000425,35775.0,0.052604,205.395617,205.347997,205.460161,205.422444,0.003022,251.038285,-0.232261


In [85]:
input_row.columns

Index(['timestamp', 'datetime', 'open', 'high', 'low', 'close', 'volume',
       'ask_price', 'ask_size', 'bid_price', 'bid_size', 'RSI', 'MACD',
       'MACD_signal', 'MACD_hist', 'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB',
       'Middle_BB', 'Lower_BB', 'ATR_1', 'ATR_2', 'ATR_5', 'ATR_10', 'ATR_20',
       'ADX', '+DI', '-DI', 'CCI', 'DLR', 'TWAP', 'VWAP', 'market_liquidity',
       'expected_price', 'log_return', 'volatility', 'mid_price', 'mean_vol',
       'mean_liq', 'transaction_cost', '5_min_volatility', '5_min_volume',
       '5_min_TC', 'forecast_6Hr_open', 'forecast_6Hr_high',
       'forecast_6Hr_low', 'forecast_6Hr_close', 'forecast_6Hr_volatility',
       'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost'],
      dtype='object')

In [87]:
len(input_row.columns)

51

Now the input row is ready to be inserted into Macrotrader

### 4. Run this input into Macrotrader and infer output

Before we can infer from the model, we need to get the inference loop in place:
1. Create a custom environment tailored for inferencing - skip rewards, etc components
2. Create a custom dynamic forecasting function to predict future feature space (depending on steps outputted by model)
3. Create a Model class for better organization and extraction of models depending on input parameters
4. Finally, we are ready to infer trade schedule from the model.

### I. Custom Environment:

### Summary of Functions

#### `__init__(self, data, scenario, action_space, preferred_timeframe=390, initial_inventory=10)`
Initializes the environment.

- **Inputs:**
  - `data (pd.DataFrame)`: Historical market data.
  - `scenario (dict)`: Scenario-specific parameters.
  - `action_space (gym.spaces)`: Action space for the agent.
  - `preferred_timeframe (int)`: Number of steps to complete the trade (default is 390).
  - `initial_inventory (int)`: Initial inventory of shares (default is 10).
- **Outputs:**
  - None

#### `reset(self)`
Resets the environment to its initial state at the beginning of a new episode.

- **Inputs:**
  - None
- **Outputs:**
  - `pd.Series`: The initial state (last row of the data).

#### `get_next_valid_market_timestamp(self, current_timestamp, time_slice_minutes)`
Calculates the next valid market timestamp based on the current timestamp and time slice.

- **Inputs:**
  - `current_timestamp (datetime)`: Current market timestamp.
  - `time_slice_minutes (int)`: Minutes to add to the current timestamp.
- **Outputs:**
  - `datetime`: The next valid market timestamp considering market hours, weekends, and holidays.

#### `step(self, action)`
Executes a step in the environment by performing the action provided by the agent.

- **Inputs:**
  - `action (np.array)`: Action array where `action[0]` is the percentage of inventory to trade and `action[1]` is the timing of the next trade.
- **Outputs:**
  - `bool`: Whether the episode is done (inventory is depleted or time is up).
  - `dict`: Additional information about the step taken.

#### `_add_noise_to_action(self, action)`
Adds noise to the agent's actions to simulate real-world uncertainties.

- **Inputs:**
  - `action (np.array)`: Action array from the agent.
- **Outputs:**
  - `np.array`: The action array with added noise.

#### `_take_action(self, size_of_slice)`
Executes the trade by reducing the inventory based on the size of the slice.

- **Inputs:**
  - `size_of_slice (int)`: Number of shares to trade.
- **Outputs:**
  - None

#### `render(self, mode='human', close=False)`
Renders the current state of the environment, including all trades executed so far.

- **Inputs:**
  - `mode (str)`: Mode of rendering (default is 'human').
  - `close (bool)`: Whether to close the render window (default is False).
- **Outputs:**
  - None

#### `print_trades(self)`
Prints a summary of all trades executed so far.

- **Inputs:**
  - None
- **Outputs:**
  - `list`: A list of dictionaries, each containing details of a trade.

In [129]:
import gym
from gym import spaces
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from datetime import timedelta

class CustomTradingEnvironment(gym.Env):
    metadata = {'render.modes': ['human']}
    
    def __init__(self, data, scenario, action_space, preferred_timeframe=390, initial_inventory=10):
        """
        Initializes the Custom Trading Environment.

        Args:
        - data (pd.DataFrame): Current input data row.
        - scenario (dict): Contains any scenario-specific parameters.
        - action_space (gym.spaces): The action space for the agent.
        - preferred_timeframe (int, optional): The number of steps in which the trade should be completed (default is 390).
        - initial_inventory (int, optional): Initial inventory of shares to be traded (default is 10).

        Returns:
        - None
        """
        super(CustomTradingEnvironment, self).__init__()
        self.data = data        
        self.preferred_timeframe = preferred_timeframe
        self.initial_inventory = initial_inventory
        self.remaining_inventory = self.initial_inventory
        self.elapsed_time = 0
        self.trades = []
        self.timestamp = pd.to_datetime(self.data['timestamp'], unit='ms')

        # State columns contain market data and technical indicators
        self.state_columns = ['open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
                              'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX', 
                              '+DI', '-DI', 'CCI', 'transaction_cost', 'forecast_6Hr_open','forecast_6Hr_close',
                              'forecast_6Hr_high','forecast_6Hr_low', 'forecast_6Hr_volatility', 
                              'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost']
        
        # Define the action space and observation space for the environment
        self.action_space = action_space
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(45,), dtype=np.float32)
        
    def reset(self):
        """
        Resets the environment to its initial state at the beginning of a new episode.

        Returns:
        - pd.Series: The last row of the data as the initial state.
        """
        print('------------------------------------------------Class resetted------------------------------------------------')
        self.remaining_inventory = self.initial_inventory
        self.elapsed_time = 0
        self.trades = []
        row_values = self.data.iloc[-1][self.state_columns].values
        state = self.data[self.state_columns].iloc[-1].values
        return state
    
    


    def get_next_valid_market_timestamp(self, current_timestamp, time_slice_minutes):
        """
        Calculates the next valid market timestamp based on the current timestamp and time slice.

        Args:
        - current_timestamp (datetime, pd.Timestamp, or numpy.int64): Current market timestamp.
        - time_slice_minutes (int): Minutes to add to the current timestamp.

        Returns:
        - datetime: The next valid market timestamp considering market hours, weekends, and holidays.
        """
        # Convert numpy.int64 (or any int) to datetime
        if isinstance(current_timestamp, (np.int64, int)):
            current_timestamp = datetime.utcfromtimestamp(current_timestamp)

        # Ensure current_timestamp is a single timestamp, not a DataFrame or Series
        if isinstance(current_timestamp, (pd.Series, pd.DataFrame)):
            current_timestamp = current_timestamp.squeeze()  # Convert to scalar if it's a Series with one element

        # Convert to datetime if it's a pandas Timestamp
        if isinstance(current_timestamp, pd.Timestamp):
            current_timestamp = current_timestamp.to_pydatetime()

        # Now it's safe to use replace
        market_start = current_timestamp.replace(hour=9, minute=30, second=0, microsecond=0)
        market_end = current_timestamp.replace(hour=16, minute=0, second=0, microsecond=0)

        # Add time slice to the current timestamp
        new_timestamp = current_timestamp + timedelta(minutes=time_slice_minutes)

        # If the new timestamp is beyond market hours
        if new_timestamp > market_end:
            # Move to the next market day's open
            new_timestamp = market_start + timedelta(days=1)

        # If the new timestamp is before market open, set it to the market start time
        if new_timestamp < market_start:
            new_timestamp = market_start

        # Skip weekends
        while new_timestamp.weekday() >= 5:  # 5 = Saturday, 6 = Sunday
            new_timestamp += timedelta(days=1)

        # Check if the new timestamp falls on a holiday
        cal = USFederalHolidayCalendar()
        holidays = cal.holidays(start=new_timestamp, end=new_timestamp + timedelta(days=365)).to_pydatetime()
        while new_timestamp in holidays:
            new_timestamp += timedelta(days=1)
            new_timestamp = new_timestamp.replace(hour=9, minute=30, second=0, microsecond=0)  # Reset to market open time

        return new_timestamp


    def step(self, action):
        """
        Executes a step in the environment by performing the action provided by the agent.

        Args:
        - action (np.array): An array where action[0] is the percentage of inventory to trade and action[1] is the timing of the next trade.

        Returns:
        - bool: Whether the episode is done (i.e., inventory is depleted or time is up).
        - dict: Additional information about the step taken.
        """
        # Adding some noise to the actions
        action = self._add_noise_to_action(action)
        print(f'Action taken: {action}')

        size_of_slice = action[0] * self.remaining_inventory
        size_of_slice = int(np.ceil(size_of_slice))

        # Scale action[1] to a desired range, e.g., 1 to 10
        timing_of_slice = int(np.ceil(action[1]))
        print(f'timing_of_slice: {timing_of_slice}')

        self.elapsed_time += timing_of_slice

        if self.elapsed_time >= self.preferred_timeframe:
            size_of_slice = self.remaining_inventory

        # Take action
        self._take_action(size_of_slice)

        current_timestamp = self.timestamp
        print(f"Current Timestamp: ", current_timestamp)
        next_timestamp = self.get_next_valid_market_timestamp(current_timestamp, timing_of_slice)

        # Print current and next step for debugging
        print(f'Current Timestamp: {current_timestamp}')
        self.elapsed_time += timing_of_slice
        print(f'Next step: {next_timestamp}')


        done = self.remaining_inventory <= 0 or self.elapsed_time >= self.preferred_timeframe

        # Record trade details
        trade_info = {
                'timestamp': next_timestamp,
                'action': action,
                'shares': size_of_slice,
                'inventory': self.remaining_inventory,
                'time left': self.preferred_timeframe - self.elapsed_time
            }
        self.trades.append(trade_info)

        info = {
            'step': next_timestamp,
            'action': action,
        }

        self.timestamp = next_timestamp

        return done, info

            
    def _add_noise_to_action(self, action):
        """
        Adds noise to the agent's actions to simulate real-world uncertainties.

        Args:
        - action (np.array): An array containing the actions from the agent.

        Returns:
        - np.array: The action array with added noise.
        """
        # Add noise to the first action (percentage of inventory)
        noise_action_0 = np.random.normal(0, 0.02, size=action[0].shape)  # Small noise for percentage
        action[0] += noise_action_0
        action[0] = np.clip(action[0], self.action_space.low[0], self.action_space.high[0])

        # Add noise to the second action (timing of next slice)
        noise_action_1 = np.random.normal(0, 5, size=action[1].shape)  # Setting SD to be 5% of the range (1-100)
        action[1] += noise_action_1
        action[1] = np.clip(action[1], self.action_space.low[1], self.action_space.high[1])

        return action

    
    def _take_action(self, size_of_slice):
        """
        Executes the trade by reducing the inventory based on the size of the slice.

        Args:
        - size_of_slice (int): The number of shares to trade.

        Returns:
        - None
        """
        self.remaining_inventory -= size_of_slice
        print(f'Remaining inventory: {self.remaining_inventory}')
        if self.remaining_inventory < 0:
            self.remaining_inventory = 0
    
    def render(self, mode='human', close=False):
        """
        Renders the current state of the environment, including all trades executed so far.

        Args:
        - mode (str, optional): The mode of rendering (default is 'human').
        - close (bool, optional): Whether to close the render window (default is False).

        Returns:
        - None
        """
        print('--------------------------------------------------')
        return self.print_trades()

    def print_trades(self):
        """
        Prints a summary of all trades executed so far.

        Returns:
        - list: A list of dictionaries, each containing details of a trade.
        """
        trades_df = pd.DataFrame(self.trades)
        for trade in self.trades:
            print(f"Timestamp: {trade['timestamp']}, Action: {trade['action']}, Shares: {trade['shares']}, Inventory: {trade['inventory']}, TimeLeft: {trade['time left']}")
        
        return self.trades


### II. Function for Dynamic Forecasting

### Function Summary

#### `dynamic_forecast(data, step)`

This function generates forecasts for various financial indicators using historical market data. It leverages time series models like Exponential Smoothing and ARIMA to predict future values of indicators such as:

- **OHLC (Open, High, Low, Close)**
- **Volume**
- **Volatility**
- **Technical Indicators** (e.g., RSI, MACD, Stochastic Oscillator)

The function considers the last few data points (based on a specified window size) to generate predictions for the next specified time steps. The forecasts are then combined into a single DataFrame that contains both the predicted future state and specific future forecasted values.

#### Parameters:
- **`data (pd.DataFrame)`**: The historical market data containing columns for various financial indicators.
- **`step (int)`**: Specifies how far into the future the forecast should be made.

#### Returns:
- **`pd.DataFrame`**: A DataFrame containing the forecasted values for the indicators, structured to include both the next step predictions and a longer-term forecast.

### Usage Example

```python
# Assuming `data` is a DataFrame containing historical market data with TC and forecasts
result = dynamic_forecast(data, step=5)
print(result.head())
```

In [151]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

def dynamic_forecast(data, step):
    """
    Generates a dynamic forecast for a set of indicators using historical data.

    The function forecasts various financial indicators such as OHLC (Open, High, Low, Close),
    volume, volatility, and technical indicators like RSI, MACD, etc., based on the past data.

    The forecasts are generated using either Exponential Smoothing or ARIMA models, depending on the indicator.

    Args:
    - data (pd.DataFrame): The historical market data.
    - step (int): The forecast step, indicating how far into the future the prediction should be made.

    Returns:
    - pd.DataFrame: A DataFrame containing the forecasted values for the last row in the dataset.
    """
    
    def forecast_last_row(data, forecast_steps, columns, window_size):
        """
        Forecasts the last row of data based on the given columns and their forecast steps.

        Args:
        - data (pd.DataFrame): The historical market data.
        - forecast_steps (dict): A dictionary specifying the forecast steps and frequency for each indicator.
        - columns (dict): A dictionary mapping each indicator to its corresponding column name in the data.
        - window_size (int): The number of past observations to consider for forecasting.

        Returns:
        - tuple: Two DataFrames, one for the row-level forecast and another for the state forecast.
        """
        last_idx = len(data) - 1
        row_forecasts = {'timestamp': data.index[last_idx]}
        state_forecasts = {}
        
        for indicator, column in columns.items():
            steps, freq = forecast_steps[indicator]
            start_idx = max(0, last_idx - window_size)
            series = data[column].iloc[start_idx:last_idx+1]
            
            # If insufficient data, skip forecast for this indicator
            if len(series) < 2:
                row_forecasts[f'forecast_{indicator}'] = None
                continue
            
            # Select appropriate model based on the indicator
            if indicator in ['open', 'high', 'low', 'close', 'RSI', 'MACD',
                           'MACD_signal', 'MACD_hist', 'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB',
                           'Middle_BB', 'Lower_BB', 'ATR_1', 'ATR_2', 'ATR_5', 'ATR_10', 'ATR_20',
                           'ADX', '+DI', '-DI', 'CCI', 'volatility','5_min_volatility', '5_min_volume']:
                model = ExponentialSmoothing(series, trend='add', seasonal=None)
            elif indicator == '5_min_TC' or indicator == 'transaction_cost':
                model = ARIMA(series, order=(5, 1, 0))
            elif indicator == 'volume' or indicator == '5_min_volume':
                shift = 1 if series.min() <= 0 else 0
                transformed_series = np.log(series + shift + 1)
                model = ExponentialSmoothing(transformed_series, trend='add', seasonal=None)
            
            # Fit the model and generate forecast
            model_fit = model.fit()
            forecast_values = model_fit.forecast(steps=steps)
            
            if indicator == 'volume':
                forecast_values = np.exp(forecast_values) - 1 - shift
                forecast_values[forecast_values < 0] = 0  # Ensure non-negative values
                
            # Add forecast to state_forecast (for specific future step)
            state_forecasts[f'{indicator}'] = forecast_values.iloc[step-1]
            
            # Add future OHLCV to row_forecasts (e.g., forecast for the last step in 6 hours)
            if indicator in ['open', 'high', 'low', 'close', 'transaction_cost', 'volume', 'volatility']:
                row_forecasts[f'forecast_6Hr_{indicator}'] = forecast_values.iloc[-1]
        
        # Create DataFrames for row-level and state forecasts
        row_forecasts_df = pd.DataFrame([row_forecasts]).reset_index(drop=True)
        state_forecasts_df = pd.DataFrame([state_forecasts]).reset_index(drop=True)
        
        return row_forecasts_df, state_forecasts_df

    # Define forecast steps and frequency for various indicators
    forecast_steps = {
        'open': (step + 360, '1T'),
        'high': (step + 360, '1T'),
        'low': (step + 360, '1T'),
        'close': (step + 360, '1T'),
        'volume': (step + 360, '1T'),
        'volatility': (step + 360, '1T'),
        'transaction_cost': (step + 360, '1T'),
        'RSI': (step, '1T'),
        'MACD': (step, '1T'),
        'MACD_signal': (step, '1T'),
        'MACD_hist': (step, '1T'),
        'Stoch_k': (step, '1T'),
        'Stoch_d': (step, '1T'),
        'OBV': (step, '1T'),
        'Upper_BB': (step, '1T'),
        'Middle_BB': (step, '1T'),
        'Lower_BB': (step, '1T'),
        'ATR_1': (step, '1T'),
        'ADX': (step, '1T'),
        '+DI': (step, '1T'),
        '-DI': (step, '1T'),
        'CCI': (step, '1T'),
        '5_min_volatility': (step, '1T'),
        '5_min_volume': (step, '1T'),
        '5_min_TC': (step, '1T') # Add bid ask forecasts as well later
    }
    
    # Map indicators to corresponding columns in the data
    columns = {col: col for col in forecast_steps}
    
    # Forecast the last row and state for the given data
    last_row_forecast, data_forecast = forecast_last_row(data, forecast_steps, columns, 300)
    
    # Combine the two DataFrames into a single input row
    input_row = pd.concat([data_forecast, last_row_forecast], axis=1)
    
    return input_row

# Usage
result = dynamic_forecast(data_testing, 360)
result.head()


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,open,high,low,close,volume,volatility,transaction_cost,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ADX,+DI,-DI,CCI,5_min_volatility,5_min_volume,5_min_TC,timestamp,forecast_6Hr_open,forecast_6Hr_high,forecast_6Hr_low,forecast_6Hr_close,forecast_6Hr_volume,forecast_6Hr_volatility,forecast_6Hr_transaction_cost
0,205.395617,205.347997,205.460161,205.422444,251.038285,0.002868,0.026863,31.346605,5.705727,1.708269,3.717775,584.379876,1077.742724,-3.652904e+06,174.075346,199.1823,223.192874,-0.006216,56.300331,6.994312,37.211492,187.49483,0.0007,-7.903364e+06,0.818392,17966,201.191235,200.955995,201.320322,201.105144,8.478631,0.002795,0.026863


In [147]:
result.columns

Index(['open', 'high', 'low', 'close', 'volume', 'volatility',
       'transaction_cost', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist',
       'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB',
       'ATR_1', 'ADX', '+DI', '-DI', 'CCI', '5_min_volatility', '5_min_volume',
       '5_min_TC', 'timestamp', 'forecast_6Hr_open', 'forecast_6Hr_high',
       'forecast_6Hr_low', 'forecast_6Hr_close', 'forecast_6Hr_volume',
       'forecast_6Hr_volatility', 'forecast_6Hr_transaction_cost'],
      dtype='object')

In [ ]:
['open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
                              'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX', 
                              '+DI', '-DI', 'CCI', 'transaction_cost', 'forecast_6Hr_open','forecast_6Hr_close',
                              'forecast_6Hr_high','forecast_6Hr_low', 'forecast_6Hr_volatility', 
                              'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost']

### III. Model Class for better organization of all the models

### Summary of Functions

#### `__init__(self)`
Initializes the `Model` class and prepares it to load various trading models based on predefined scenarios.

- **Inputs:**
  - None
- **Outputs:**
  - None

#### `_load_model(self, env, filepath, policy_kwargs, best_hyperparameters)`
Loads a pre-trained PPO model using a custom transformer policy and predefined hyperparameters.

- **Inputs:**
  - `env (gym.Env)`: The trading environment to use with the model.
  - `filepath (str)`: The path to the file containing the saved model parameters.
  - `policy_kwargs (dict)`: Additional keyword arguments related to the policy configuration.
  - `best_hyperparameters (dict)`: The best hyperparameters to use for the PPO model.
- **Outputs:**
  - `model`: The loaded PPO model configured with the specified environment and hyperparameters.

#### `_get_small(self, data)`
Sets up and returns a model for small trade scenarios.

- **Inputs:**
  - `data (pd.DataFrame)`: Historical market data to be used in the trading environment.
- **Outputs:**
  - `model`: The PPO model configured for small trade scenarios.

#### `_get_small_medium(self, data)`
Sets up and returns a model for small to medium trade scenarios.

- **Inputs:**
  - `data (pd.DataFrame)`: Historical market data to be used in the trading environment.
- **Outputs:**
  - `model`: The PPO model configured for small to medium trade scenarios.

#### `_get_medium(self, data)`
Sets up and returns a model for medium trade scenarios.

- **Inputs:**
  - `data (pd.DataFrame)`: Historical market data to be used in the trading environment.
- **Outputs:**
  - `model`: The PPO model configured for medium trade scenarios.

#### `_get_medium_large(self, data)`
Sets up and returns a model for medium to large trade scenarios.

- **Inputs:**
  - `data (pd.DataFrame)`: Historical market data to be used in the trading environment.
- **Outputs:**
  - `model`: The PPO model configured for medium to large trade scenarios.

#### `_get_large(self, data)`
Sets up and returns a model for large trade scenarios.

- **Inputs:**
  - `data (pd.DataFrame)`: Historical market data to be used in the trading environment.
- **Outputs:**
  - `model`: The PPO model configured for large trade scenarios.

#### Usage:

```
python
# Initialize the model manager
model_manager = Model()

# Load the small trade model using historical data
small_model = model_manager._get_small(data)

# Similarly, you can load models for different trade sizes
medium_model = model_manager._get_medium(data)
```

In [156]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Model:
    """
    This class is responsible for loading pre-trained models for different trading scenarios. 
    Each method in the class loads a model configured with specific hyperparameters and action spaces 
    based on the trading scenario (small, small-medium, medium, medium-large, large).
    """

    def __init__(self):
        """
        Initializes the Model class.
        """
        
        
    def _load_model(self, env, filepath, policy_kwargs, best_hyperparameters):
        """
        Loads a pre-trained PPO model with a custom transformer policy from a specified file path.
        
        Args:
        - env (gym.Env): The trading environment for the model.
        - filepath (str): The file path to the saved model parameters.
        - policy_kwargs (dict): Additional keyword arguments for configuring the policy.
        - best_hyperparameters (dict): The best hyperparameters used for training the PPO model.

        Returns:
        - model (PPO): The loaded PPO model configured with the given environment and hyperparameters.
        """
        model = PPO(CustomTransformerPolicy, env, verbose=1, policy_kwargs=policy_kwargs, **best_hyperparameters)
        # Load the model parameters into the new model
        model.policy.load_state_dict(torch.load(filepath))
        print(f"Model loaded from {filepath}")
        return model
        
    def _get_small(self, data):
        """
        Configures and returns a PPO model for small trade scenarios.

        Args:
        - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

        Returns:
        - model (PPO): The PPO model configured for small trade scenarios.
        """
        file_path = 'Models/model_small.h5'
        action_space = spaces.Box(low=np.array([0.33, 30]), high=np.array([1, 50]), dtype=np.float32)
        env = TradingEnvironment(data, action_space, preferred_timeframe=390, initial_inventory=10, scenario='small')
        scenario = 'small'
        preferred_timeframe = 390
        initial_inventory = 10
        best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
        model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
        return model
    
    def _get_small_medium(self, data):
        """
        Configures and returns a PPO model for small to medium trade scenarios.

        Args:
        - data (pd.DataFrame): Historical market data to be used in the trading environmen - 1 row.

        Returns:
        - model (PPO): The PPO model configured for small to medium trade scenarios.
        """
        file_path = 'Models/model_small_med.h5'
        preferred_timeframe = 390
        initial_inventory = 100
        action_space = spaces.Box(low=np.array([0.2, 20]), high=np.array([0.66, 30]), dtype=np.float32)
        env = TradingEnvironment(data,action_space, preferred_timeframe=preferred_timeframe, initial_inventory=initial_inventory, scenario = 'small-medium')
        scenario = 'small-medium'
        best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
        model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
        return model
    
    def _get_medium(self, data):
        """
        Configures and returns a PPO model for medium trade scenarios.

        Args:
        - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

        Returns:
        - model (PPO): The PPO model configured for medium trade scenarios.
        """
        file_path = 'Models/model_med.h5'
        preferred_timeframe = 390
        initial_inventory = 500
        action_space = spaces.Box(low=np.array([0.20, 30]), high=np.array([0.50, 50]), dtype=np.float32)
        env = TradingEnvironment(data, action_space, preferred_timeframe=preferred_timeframe, initial_inventory=initial_inventory,scenario = 'medium')
        scenario = 'medium'
        best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
        model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
        return model
    
    def _get_medium_large(self, data):
        """
        Configures and returns a PPO model for medium to large trade scenarios.

        Args:
        - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

        Returns:
        - model (PPO): The PPO model configured for medium to large trade scenarios.
        """
        file_path = 'Models/model_med_lg.h5'
        preferred_timeframe = 390
        initial_inventory = 2000
        action_space = spaces.Box(low=np.array([0.10, 30]), high=np.array([0.40, 50]), dtype=np.float32)
        env = TradingEnvironment(data, action_space, preferred_timeframe=preferred_timeframe, initial_inventory=initial_inventory, scenario = 'medium-large')
        scenario = 'medium-large'
        best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
        model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
        return model
    
    def _get_large(self, data):
        """
        Configures and returns a PPO model for large trade scenarios.

        Args:
        - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

        Returns:
        - model (PPO): The PPO model configured for large trade scenarios.
        """
        file_path = 'Models/model_lg.h5'
        preferred_timeframe = 390
        initial_inventory = 10000
        action_space = spaces.Box(low=np.array([0.05, 30]), high=np.array([0.33, 50]), dtype=np.float32)
        env = TradingEnvironment(data, action_space, preferred_timeframe=preferred_timeframe, initial_inventory=initial_inventory, scenario = 'large')
        scenario = 'large'
        best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
        model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
        return model


### IV. Infer the Model - Last Step

In [157]:
def infer_macro(timeframe, transaction_size, input_data, Model):
    # First decide which model to use
    meta = MetaLearner()
    scenario = meta.classify_scenario(transaction_size, timeframe)
    print(f"Scenario: {scenario}")
    
    models = Model()
    
    if scenario == 'small':
        model = models._get_small(input_data)
    elif scenario == 'small-medium':
        model = models._get_small_medium(input_data)
    elif scenario == 'medium':
        model = models._get_medium(input_data)
    elif scenario == 'medium-large':
        model = models._get_medium_large(input_data)
    elif scenario == 'large':
        model = models._get_large(input_data)
    
    # Now we have the model we want to use - we can start testing loop
    action_space = spaces.Box(low=np.array([0.10, 30]), high=np.array([0.40, 50]), dtype=np.float32)
    
    env = CustomTradingEnvironment(input_data,scenario,action_space, preferred_timeframe=timeframe, initial_inventory=transaction_size)
    obs = env.reset()
    obs = obs.astype(np.float32)
    done = False
    forecast_step = 0
    observations = []
    state_cols = ['open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
                              'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX', 
                              '+DI', '-DI', 'CCI', 'transaction_cost', 'forecast_6Hr_open','forecast_6Hr_close',
                              'forecast_6Hr_high','forecast_6Hr_low', 'forecast_6Hr_volatility', 
                              'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost']

    while not done:
        print(1)
        action, _states = model.predict(obs)
        step = int(np.ceil(action[1]))
        forecast_step += step
        # Now, forecast for this step in the future
        obs = dynamic_forecast(data_testing, forecast_step)
        obs = obs.squeeze()
        observations.append(obs)
        obs = obs[state_cols]
        # Run the step function to reflect inventory and elapsed time
        done, info = env.step(action)
        
        if done:
            break
    
    micro_input = pd.DataFrame(observations)
    trades = env.render()
    return trades, micro_input


trades, micro_input = infer_macro(390, 500, input_row, Model)

Scenario: medium-large
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Model loaded from Models/model_med_lg.h5
------------------------------------------------Class resetted------------------------------------------------
1


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Action taken: [ 0.13938463 40.237743  ]
timing_of_slice: 41
Remaining inventory: 430
Current Timestamp:  0   2024-08-07 20:12:00
Name: timestamp, dtype: datetime64[ns]
Current Timestamp: 0   2024-08-07 20:12:00
Name: timestamp, dtype: datetime64[ns]
Next step: 2024-08-08 09:30:00
1


...

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Action taken: [ 0.10534527 30.        ]
timing_of_slice: 30
Remaining inventory: 117
Current Timestamp:  2024-08-08 11:05:00
Current Timestamp: 2024-08-08 11:05:00
Next step: 2024-08-08 11:35:00
1


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


Action taken: [ 0.11118661 30.        ]
timing_of_slice: 30
Remaining inventory: 103
Current Timestamp:  2024-08-08 11:35:00
Current Timestamp: 2024-08-08 11:35:00
Next step: 2024-08-08 12:05:00
--------------------------------------------------
Timestamp: 2024-08-08 09:30:00, Action: [ 0.13938463 40.237743  ], Shares: 70, Inventory: 430, TimeLeft: 308
Timestamp: 2024-08-08 10:00:00, Action: [ 0.39074886 30.        ], Shares: 169, Inventory: 261, TimeLeft: 248
Timestamp: 2024-08-08 10:30:00, Action: [ 0.15513323 30.        ], Shares: 41, Inventory: 220, TimeLeft: 188
Timestamp: 2024-08-08 11:05:00, Action: [ 0.4      34.087597], Shares: 89, Inventory: 131, TimeLeft: 118
Timestamp: 2024-08-08 11:35:00, Action: [ 0.10534527 30.        ], Shares: 14, Inventory: 117, TimeLeft: 58
Timestamp: 2024-08-08 12:05:00, Action: [ 0.11118661 30.        ], Shares: 14, Inventory: 103, TimeLeft: -2


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [158]:
micro_input

,open,high,low,close,volume,volatility,transaction_cost,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ADX,+DI,-DI,CCI,5_min_volatility,5_min_volume,5_min_TC,timestamp,forecast_6Hr_open,forecast_6Hr_high,forecast_6Hr_low,forecast_6Hr_close,forecast_6Hr_volume,forecast_6Hr_volatility,forecast_6Hr_transaction_cost
0,209.249635,209.374000,209.255013,209.379970,5097.714127,0.002935,0.027289,44.325126,0.319663,-0.032803,0.329160,97.328996,138.171962,1.106260e+07,207.145905,208.8579,210.478530,0.144016,30.573301,14.385289,27.875442,53.835611,0.000448,-6.259115e+05,0.617347,17966.0,205.045252,204.981997,205.115174,205.062669,190.751944,0.002862,0.026863
0,208.899270,209.008000,208.910027,209.020195,3878.126319,0.002929,0.026864,43.145260,0.809306,0.125476,0.637216,141.606349,223.587485,9.724829e+06,204.139490,207.9783,211.634380,0.130358,32.912122,13.713382,28.724174,65.986449,0.000471,-1.287498e+06,1.168218,17966.0,204.694887,204.615997,204.770188,204.702894,144.885804,0.002856,0.026863
0,208.548904,208.641999,208.565040,208.660420,2950.258028,0.002923,0.026863,41.965395,1.298948,0.283755,0.945272,185.883701,309.003009,8.387055e+06,201.133076,207.0987,212.790229,0.116701,35.250943,13.041475,29.572906,78.137287,0.000494,-1.949085e+06,0.997831,17966.0,204.344522,204.249997,204.425201,204.343119,109.990624,0.002850,0.026863
0,208.198539,208.275999,208.220054,208.300644,2244.331354,0.002917,0.026863,40.785529,1.788590,0.442035,1.253328,230.161054,394.418533,7.049282e+06,198.126662,206.2191,213.946078,0.103043,37.589763,12.369568,30.421638,90.288125,0.000517,-2.610671e+06,0.695206,17966.0,203.994157,203.883996,204.080215,203.983344,83.442203,0.002844,0.026863
0,207.848174,207.909999,207.875067,207.940869,1707.258932,0.002911,0.026863,39.605664,2.278232,0.600314,1.561383,274.438407,479.834057,5.711509e+06,195.120247,205.3395,215.101928,0.089386,39.928584,11.697661,31.270369,102.438963,0.000540,-3.272258e+06,0.701188,17966.0,203.643791,203.517996,203.735228,203.623569,63.244035,0.002838,0.026863
0,207.497809,207.543999,207.530081,207.581094,1298.651641,0.002905,0.026863,38.425798,2.767874,0.758593,1.869439,318.715759,565.249581,4.373736e+06,192.113833,204.4599,216.257777,0.075728,42.267405,11.025754,32.119101,114.589801,0.000563,-3.933845e+06,0.841006,17966.0,203.293426,203.151996,203.390242,203.263794,47.877172,0.002832,0.026863
